# 🔍 Trực quan hóa Bounding Box & Dữ liệu Dự án YOLO11n
Notebook này hỗ trợ kiểm tra trực quan toàn bộ kết quả phát hiện và theo dõi:
1. **Thư mục `output/`**: Chứa các cặp **ảnh gốc**, file **track xe (`_vehicle.txt`)** và file **track biển số (`_plate.txt`)** sau khi chạy `auto_track_label.py`.
2. **Thư mục `data_source/Vietnam license-plate.v1i.yolov8/`**: Dữ liệu mẫu biển số xe dùng để huấn luyện.

In [ ]:
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from ultralytics import YOLO

# 1. Tự động xác định thư mục gốc của project
current_dir = Path.cwd()
if (current_dir / "output").exists() or (current_dir / "data_source").exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / "output").exists() or (current_dir.parent / "data_source").exists():
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

OUTPUT_DIR = PROJECT_ROOT / "output"
DATA_SOURCE_DIR = PROJECT_ROOT / "data_source"
MODEL_PATH = PROJECT_ROOT / "yolo11n.pt"
BEST_PLATE_PATH = PROJECT_ROOT / "runs" / "train" / "yolo11n_license_plate" / "weights" / "best.pt"

# Tên các class xe COCO
CLASS_NAMES_VEHICLE = {2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck', 0: 'person'}
# Tên class biển số
CLASS_NAMES_PLATE = {0: 'license_plate'}

print(f"✓ Thư mục Project:  {PROJECT_ROOT.resolve()}")
print(f"✓ Thư mục Output:   {OUTPUT_DIR.resolve()}")
print(f"✓ Model Xe:         {MODEL_PATH.resolve()}")
print(f"✓ Model Biển số:    {BEST_PLATE_PATH.resolve()} (Tồn tại: {BEST_PLATE_PATH.exists()})")

In [ ]:
def parse_yolo_txt(txt_path, img_w, img_h, names_dict, default_name="obj"):
    """
    Đọc file nhãn YOLO .txt (hỗ trợ cả format 5 số hoặc 6 số có track_id).
    """
    boxes_data = []
    if not txt_path or not Path(txt_path).exists():
        return boxes_data

    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                cls_id = int(float(parts[0]))
                xc, yc, w, h = map(float, parts[1:5])
                track_id = int(parts[5]) if len(parts) >= 6 else None

                xmin = int(round((xc - w / 2) * img_w))
                ymin = int(round((yc - h / 2) * img_h))
                xmax = int(round((xc + w / 2) * img_w))
                ymax = int(round((yc + h / 2) * img_h))

                xmin = max(0, min(img_w - 1, xmin))
                ymin = max(0, min(img_h - 1, ymin))
                xmax = max(0, min(img_w - 1, xmax))
                ymax = max(0, min(img_h - 1, ymax))

                box_pixel_w = xmax - xmin
                box_pixel_h = ymax - ymin
                area = box_pixel_w * box_pixel_h
                area_ratio = (area / (img_w * img_h)) * 100

                boxes_data.append({
                    "cls_id": cls_id,
                    "cls_name": names_dict.get(cls_id, f"{default_name}_{cls_id}"),
                    "track_id": track_id,
                    "pixel": (xmin, ymin, xmax, ymax),
                    "pixel_dim": (box_pixel_w, box_pixel_h),
                    "area_ratio": area_ratio,
                    "norm": (xc, yc, w, h)
                })
    return boxes_data

def draw_boxes_custom(img_bgr, boxes_data, box_color, label_prefix=""):
    """
    Vẽ bounding box lên ảnh với màu sắc chỉ định.
    """
    for box in boxes_data:
        xmin, ymin, xmax, ymax = box["pixel"]
        track_id = box["track_id"]
        cls_name = box["cls_name"]
        
        cv2.rectangle(img_bgr, (xmin, ymin), (xmax, ymax), box_color, 2)
        
        if track_id is not None and track_id != -1:
            text = f"{label_prefix}{cls_name} #{track_id}"
        else:
            text = f"{label_prefix}{cls_name}"
            
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(img_bgr, (xmin, max(0, ymin - th - 5)), (xmin + tw + 2, ymin), box_color, -1)
        cv2.putText(img_bgr, text, (xmin + 1, max(th, ymin - 3)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1, cv2.LINE_AA)
    return img_bgr

### 📌 Phần 2: Xem kết quả trong thư mục `output/` (Đồng thời Track Xe & Track Biển Số)

In [ ]:
def inspect_output_results(target_folder_name=None, frame_idx=0):
    """
    Xem kết quả trực tiếp từ thư mục output/:
    - Đọc ảnh gốc
    - Đọc file track xe (_vehicle.txt)
    - Đọc file track biển số (_plate.txt)
    """
    if not OUTPUT_DIR.exists():
        print(f"❌ Thư mục {OUTPUT_DIR} chưa tồn tại!")
        return
        
    subdirs = sorted([d for d in OUTPUT_DIR.iterdir() if d.is_dir()])
    
    if not subdirs:
        print(f"⚠️ Thư mục '{OUTPUT_DIR.resolve()}' hiện đang trống!")
        print("👉 Vui lòng chạy lệnh: python auto_track_label.py để xuất kết quả vào output/")
        return
        
    # Chọn folder clip
    if target_folder_name:
        target_dir = OUTPUT_DIR / str(target_folder_name)
    else:
        target_dir = subdirs[0]
        
    if not target_dir.exists():
        print(f"Không tìm thấy folder: {target_dir}")
        return
        
    valid_exts = (".jpg", ".png", ".jpeg")
    img_files = sorted([f for f in target_dir.iterdir() if f.suffix.lower() in valid_exts and not f.stem.endswith("_annotated")])
    
    if not img_files:
        print(f"Không có ảnh trong: {target_dir}")
        return
        
    img_file = img_files[min(frame_idx, len(img_files) - 1)]
    mat = cv2.imread(str(img_file))
    if mat is None:
        print(f"Không đọc được ảnh: {img_file}")
        return
        
    h, w = mat.shape[:2]
    
    # Tìm file nhãn xe và biển số tương ứng
    veh_txt = target_dir / f"{img_file.stem}_vehicle.txt"
    plate_txt = target_dir / f"{img_file.stem}_plate.txt"
    fallback_txt = target_dir / f"{img_file.stem}.txt"
    
    # Đọc dữ liệu
    if veh_txt.exists():
        veh_boxes = parse_yolo_txt(veh_txt, w, h, CLASS_NAMES_VEHICLE, "vehicle")
    elif fallback_txt.exists():
        veh_boxes = parse_yolo_txt(fallback_txt, w, h, CLASS_NAMES_VEHICLE, "vehicle")
    else:
        veh_boxes = []
        
    plate_boxes = parse_yolo_txt(plate_txt, w, h, CLASS_NAMES_PLATE, "plate") if plate_txt.exists() else []
    
    # Vẽ bounding box: Xe màu xanh lá, Biển số màu đỏ/cam
    drawn = mat.copy()
    draw_boxes_custom(drawn, veh_boxes, (0, 200, 0), label_prefix="🚗 ")
    draw_boxes_custom(drawn, plate_boxes, (0, 69, 255), label_prefix="🏷️ ")
    
    # Hiển thị ảnh song song
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(cv2.cvtColor(mat, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f"Ảnh gốc: {img_file.name} ({w} x {h} px)")
    axes[0].axis("off")
    
    axes[1].imshow(cv2.cvtColor(drawn, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Kết quả: 🚗 {len(veh_boxes)} Xe | 🏷️ {len(plate_boxes)} Biển số")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()
    
    # In bảng chi tiết thông số
    print("=" * 105)
    print(f"THƯ MỤC OUTPUT: {target_dir.name} | FILE: {img_file.name}")
    print("=" * 105)
    all_boxes = [("XE", b) for b in veh_boxes] + [("BIỂN SỐ", b) for b in plate_boxes]
    if not all_boxes:
        print("-> Không có đối tượng nào trong frame này.")
    else:
        header = f"{'Loại':<10} | {'Class (ID)':<16} | {'Track ID':<10} | {'Pixel Box [xmin, ymin, xmax, ymax]':<35} | {'Kích thước box':<16}"
        print(header)
        print("-" * 105)
        for category, b in all_boxes:
            cls_str = f"{b['cls_name']} ({b['cls_id']})"
            track_str = str(b['track_id']) if b['track_id'] is not None else "-"
            pixel_str = f"[{b['pixel'][0]}, {b['pixel'][1]}, {b['pixel'][2]}, {b['pixel'][3]}]"
            size_str = f"{b['pixel_dim'][0]} x {b['pixel_dim'][1]} px"
            print(f"{category:<10} | {cls_str:<16} | {track_str:<10} | {pixel_str:<35} | {size_str:<16}")
    print("=" * 105)

# Chạy kiểm tra kết quả trong output
inspect_output_results()

### 📌 Phần 3: Xem danh sách tổng quan tất cả thư mục trong `output/`

In [ ]:
if not OUTPUT_DIR.exists():
    print("Thư mục output chưa tồn tại.")
else:
    subdirs = sorted([d for d in OUTPUT_DIR.iterdir() if d.is_dir()])
    print(f"Tổng số thư mục clip đã xử lý trong output/: {len(subdirs)}\n")
    header = f"{'STT':<4} | {'Thư mục Clip':<45} | {'Số lượng ảnh':<15}"
    print(header)
    print("-" * 70)
    for i, d in enumerate(subdirs[:15], 1):
        imgs = list(d.glob("*.jpg")) + list(d.glob("*.png"))
        print(f"{i:<4} | {d.name:<45} | {len(imgs):<15}")
    if len(subdirs) > 15:
        print(f"... và {len(subdirs) - 15} thư mục khác.")